# 01 — Data preparation

Builds the fixed 500-1000 example subset of `glaiveai/glaive-function-calling-v2` and freezes the 80/20 train/test split. CPU-only — can also be run locally instead of in Colab; included here for convenience since Colab is the main workflow.

See `configs/data.yaml` for filtering parameters and `src/adbench/data/prepare.py` for the implementation (TODO).

In [4]:
# Public repo — no auth needed to clone. Skips re-cloning if this
# session's runtime already has the repo (e.g. you ran
# 00_setup_colab.ipynb earlier in this same session).
import os
import sys

# Reduces CUDA OOM from a single large allocation (e.g. loading the 14B
# teacher) by letting the allocator grow a segment incrementally instead of
# needing one big contiguous block upfront.
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

ON_KAGGLE = os.path.isdir('/kaggle')
REPO_DIR = '/kaggle/working/agentic-distillation-benchmark' if ON_KAGGLE else '/content/agentic-distillation-benchmark'

if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/Nahla-Nabil/agentic-distillation-benchmark.git {REPO_DIR}

os.chdir(REPO_DIR)
!pip install -q -r requirements-colab.txt

# Put src/ on sys.path (for `import adbench` right here in this kernel)
# AND on PYTHONPATH (for `!python -m adbench...` subprocess calls in later
# cells, which inherit the environment but not this process's sys.path)
# instead of `pip install -e .` — an editable install registers itself via
# a .pth file that Python's site module only reads at interpreter startup,
# so `import adbench` fails with ModuleNotFoundError in this same
# still-running kernel until you restart it. Both of the below work
# immediately, no restart needed, on Colab or Kaggle.
src_path = os.path.join(REPO_DIR, 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)
os.environ['PYTHONPATH'] = src_path + os.pathsep + os.environ.get('PYTHONPATH', '')

In [ ]:
# Re-run this any time (after I've pushed a fix) to sync this session's
# cloned repo to the latest on GitHub, without re-cloning or restarting.
!cd {REPO_DIR} && git checkout -- . && git pull

In [5]:

!python -m adbench.data.prepare --config configs/data.yaml

Loading glaiveai/glaive-function-calling-v2 ...
README.md: 100%|████████████████████████████████| 106/106 [00:00<00:00, 736kB/s]
glaive-function-calling-v2.json: 100%|████████| 271M/271M [00:02<00:00, 132MB/s]
Generating train split: 100%|█| 112960/112960 [00:02<00:00, 54525.52 examples/s]
Loaded 112960 raw rows.
  ...0/112960
  ...20000/112960
  ...40000/112960
  ...60000/112960
  ...80000/112960
  ...100000/112960

=== prepare.py summary ===
raw rows processed: 112960
drop reasons: {'zero_calls': 49742, 'multi_call': 19583, 'tool_not_selected': 29073, 'ok': 10126, 'non_canonical_args': 3437, 'parse_error': 999}
distinct tool names seen in raw data: 966
kept examples per selected tool (before the per-tool cap):
  calculate_age: 1722
  calculate_bmi: 2946
  calculate_discount: 1852
  calculate_distance: 189
  calculate_tip: 1913
  convert_currency: 1081
  generate_random_number: 240
  get_stock_price: 183
train: 640   test: 160   total: 800
wrote: /kaggle/working/agentic-distillation-b

## Inspect the resulting subset

TODO once prepare.py exists: load train/test.jsonl, print example count, tool-type distribution, and a couple of sample tool-calling conversations, as a sanity check before training.